# 🔧 Foundations
> Utilities to help with integration of BeerCSS with FastHTML

In [ ]:
#| default_exp foundations

In [ ]:
#| export

from typing import Any, Iterable, Optional, Union
from enum import Enum
from fastcore.utils import *
from nbdev.showdoc import show_doc

## 📦 VEnum

Base class for creating enums that work seamlessly with FastHTML components.

| Feature | Description |
|---------|-------------|
| `__str__` | Returns the enum's value as a string |
| `__add__` / `__radd__` | Concatenates with other values via `stringify` |

> 💡 **Use case**: Define CSS class enums that can be combined with `+` operator.

In [ ]:
#| export
#| code-fold: true

class VEnum(Enum):
    """Enum with string conversion and concatenation support"""
    def __str__(self): return self.value
    def __add__(self, other): return stringify((self, other))
    def __radd__(self, other): return stringify((other, self))

In [ ]:

show_doc(VEnum)

In [ ]:
#| eval: false

# Example: Define a Size enum
class Size(VEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"

# Concatenate enums with strings
Size.SMALL + "primary"  # Returns: "small primary"
"btn " + Size.LARGE     # Returns: "btn large"

## 🔄 stringify

Converts various input types into space-separated strings for FT component class attributes.

| Input | Output |
|-------|--------|
| `None` / empty list | `""` |
| Single value | `str(value)` |
| List/tuple | Space-joined string |

In [ ]:
#| export
#| code-fold: true

def stringify(o):
    """Converts input types into strings that can be passed to FT components"""
    if is_listy(o): 
        return ' '.join(map(str, o)) if o else ""
    return str(o)

In [ ]:
show_doc(stringify)

In [ ]:
#| eval: false

stringify(["primary", "large", "rounded"])  # "primary large rounded"
stringify(Size.SMALL)                        # "small"
stringify((Size.MEDIUM, "filled"))           # "medium filled"

## 🧹 Token Utilities

Functions for normalizing and deduplicating CSS class tokens.

| Function | Purpose |
|----------|---------|
| `normalize_tokens` | Converts mixed inputs (strings, enums, lists) into a flat list of tokens |
| `dedupe_preserve_order` | Removes duplicate tokens while keeping original order |

> 💡 **Use case**: Clean up class attributes before passing to components — handles `"btn primary"`, `[Size.LARGE, "btn"]`, or mixed inputs.

In [ ]:
#| export
#| code-fold: true

def normalize_tokens(cls):
    """Normalize class input to list of string tokens"""
    if cls is None:
        return []
    if isinstance(cls, str):
        return cls.split()
    if isinstance(cls, Enum):
        return [str(cls)]
    if is_listy(cls):
        tokens = []
        for item in cls:
            if isinstance(item, str):
                tokens.extend(item.split())
            elif isinstance(item, Enum):
                tokens.append(str(item))
        return tokens
    return []


def dedupe_preserve_order(tokens):
    """Remove duplicates while preserving order"""
    seen = set()
    result = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            result.append(token)
    return result

In [ ]:
show_doc(normalize_tokens)

In [ ]:
show_doc(dedupe_preserve_order)

In [ ]:
#| eval: false

# Pipeline: normalize → dedupe → stringify
tokens = normalize_tokens(["btn primary", Size.LARGE, "primary"])  # ['btn', 'primary', 'large', 'primary']
unique = dedupe_preserve_order(tokens)                              # ['btn', 'primary', 'large']
stringify(unique)                                                   # "btn primary large"

In [ ]:
#| hide

import nbdev as nb
nb.nbdev_export()